# Day 21 — Weekly Review: Week 3, SQL and Postgres (14 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: check Week 3 against what your repo shows, decide what has to be finished before FastAPI, and set up Week 4, the last week of Phase 1.

- Deliverable: day21/day21_notes.ipynb with your checklist, predictions and recap answers filled in; Day 4 finished and pushed; Day 15's schema.sql and queries.sql in the repo; the SQL coding mock done in the project chat.

- Where this sits: end of Week 3, Phase 1 (Foundation). Week 4 builds a FastAPI service on top of the Postgres skills from this week and closes Phase 1 with a working REST API backed by a database.

Time split (5h): a review day is a full working day. Most of it goes into the oldest open work, because re-reading notes changes nothing in the repo.

| Block | Time | What you do |
|---|---|---|
| Review | 1h | Recap, Week 3 table, checklist, mixed review, scores and plan |
| Catch-up | 2.5h | Finish Day 4 and push it; then Day 15's schema.sql and queries.sql |
| DSA | 0.5h | Day 4's top words and balanced brackets |
| Career | 1h | SQL coding mock in chat (40 min), LinkedIn action (20 min) |

Open items first: the repo on GitHub still has one commit from 28 Oct. Days 1 to 3 are finished, Day 4 is starter files with its functions still raising NotImplementedError, and nothing from Days 5 to 20 is there. The starter folders for those days reached you as patch and zip files, because the scheduled task that writes these sessions is not allowed to push to your repo. If you have been working on your laptop, today is the day to push all of it, finished or not.

## Recap (2 minutes, from memory)

- You need the top 2 entries by views in each content type. Why can't you write WHERE row_number() OVER (...) <= 2 in the same SELECT, and what do you do instead?

- What does lag(views) OVER (PARTITION BY uid ORDER BY day) return on the first row of each entry, and how do you show 0 there instead?

- In report.py you pass a date to pd.read_sql() with text() and :start. Why is that better than putting the date into the SQL string with an f-string?

Answers are on the last page of today's session.

## Week 3 in one table

Six days, six habits. Say the right-hand column out loud without looking. Any row you cannot explain to a colleague in a minute goes on your revision list.

| Day | Big idea | The line to remember |
|---|---|---|
| 15 | Tables, keys, types, CREATE TABLE, INSERT, SELECT | SQL says what you want, not how; money is numeric, time is timestamptz |
| 16 | Aggregates, GROUP BY, HAVING, NULL, CASE | WHERE filters rows, HAVING filters groups; NULL is unknown, not zero |
| 17 | Keys, joins and normalisation | One fact in one place; a LEFT JOIN keeps rows with no match |
| 18 | psycopg 3, parameters, transactions | Never build SQL with f-strings; one load is one transaction |
| 19 | Constraints, upserts, indexes, EXPLAIN | Let the database enforce rules; measure with EXPLAIN ANALYZE before adding an index |
| 20 | CTEs, window functions, JSONB, pandas | Window functions rank and compare without losing rows |

The thread through the week: almost every customer system you will connect to keeps its data in a relational database, and most AI features read from one. Week 3 took your CMS export from a CSV file into Postgres, made the load safe to run every night, and answered content questions in SQL. In Week 4 that database gets an API in front of it, which is the shape of most FDE integration work.

## Self-assessment checklist

Tick only what you can do without notes or search. Copy the list into the notebook and mark each line yes or no. A no is useful: it tells us exactly what to revise.

- Create a table with an identity primary key, NOT NULL, a DEFAULT and a CHECK, and explain why money is numeric and not float.

- Write a SELECT with WHERE, ORDER BY ... NULLS LAST and LIMIT/OFFSET, and say why AND binds tighter than OR.

- Explain the difference between count(*) and count(column), and between WHERE and HAVING.

- Explain why x NOT IN (1, NULL) returns no rows, and fix it with NOT EXISTS.

- Model a many-to-many relationship (entries and tags) with a junction table and a composite primary key.

- Write a LEFT JOIN that finds entries with no tags (an anti-join).

- Run a parameterised query from Python with psycopg and explain how it stops SQL injection.

- Load many rows in one transaction, and say what happens to the database when row 4,000 of 5,000 fails.

- Write an upsert with ON CONFLICT ... DO UPDATE and explain what EXCLUDED is.

- Read an EXPLAIN ANALYZE plan well enough to say whether an index was used.

- Find the latest version per entry with row_number() in a CTE.

- Read a value out of a JSONB column with ->> and cast it to a number.

## Check your understanding (predict first, then run)

Seven questions from across the week, mixed up on purpose. Write every prediction in the notebook before you run anything. If one surprises you, find the line that caused it and explain it in one sentence. Questions 1 to 6 are one SQL file: save it as day21/review.sql and run it with psql -d postgres -f review.sql. It uses TEMP tables, which disappear when psql exits, so it cannot damage your real tables.

```sql
-- Week 3 mixed review: predict each numbered result, then run the file.
-- TEMP tables vanish when psql exits, so your real tables are safe.
CREATE TEMP TABLE entries (
    id      int  PRIMARY KEY,
    type    text NOT NULL,
    locale  text,           -- NULL here means "locale not set yet"
    views   int,            -- NULL means "no analytics data yet"
    status  text NOT NULL
);
INSERT INTO entries VALUES
    (1, 'blog', 'en-us', 120,  'published'),
    (2, 'blog', 'hi-in', NULL, 'draft'),
    (3, 'page', NULL,    40,   'published'),
    (4, 'page', 'en-us', 40,   'published'),
    (5, 'faq',  'en-us', NULL, 'draft');

-- 1 (Day 16) count(*) vs count(column), and what avg() does with NULL
SELECT count(*) AS all_rows, count(views) AS with_views,
       round(avg(views)) AS avg_views
FROM entries;

-- 2 (Day 16) a list with a NULL in it
SELECT count(*) FROM entries WHERE locale NOT IN ('hi-in', NULL);

-- 3 (Day 17) LEFT JOIN, then count per type
CREATE TEMP TABLE entry_tags (
    entry_id int REFERENCES entries (id),
    tag      text
);
INSERT INTO entry_tags VALUES (1, 'sale'), (1, 'diwali'), (3, 'sale');
SELECT e.type, count(t.tag) AS tags, count(*) AS row_count
FROM entries AS e
LEFT JOIN entry_tags AS t ON t.entry_id = e.id
GROUP BY e.type
ORDER BY e.type;

-- 4 (Day 20) ranking with a tie
SELECT id, views,
       rank() OVER (ORDER BY views DESC) AS rnk
FROM entries
WHERE status = 'published'
ORDER BY rnk, id;

-- 5 (Day 19) the same upsert, run twice
INSERT INTO entries VALUES (4, 'page', 'en-us', 99, 'published')
ON CONFLICT (id) DO UPDATE SET views = EXCLUDED.views
WHERE entries.views IS DISTINCT FROM EXCLUDED.views
RETURNING id, views;
INSERT INTO entries VALUES (4, 'page', 'en-us', 99, 'published')
ON CONFLICT (id) DO UPDATE SET views = EXCLUDED.views
WHERE entries.views IS DISTINCT FROM EXCLUDED.views
RETURNING id, views;

-- 6 (Day 15) AND and OR without brackets
SELECT id FROM entries
WHERE type = 'blog' OR type = 'page' AND views > 50
ORDER BY id;
```

Question 7 is about psycopg. Save it as day21/q7.py and run it with your Day 18 virtual environment active:

In [ ]:
# Question 7 (Day 18): predict what each block prints, then run it.
import psycopg

# use your own connection string here (the DATABASE_URL from Day 18)
URL = "postgresql://postgres@localhost/postgres"

with psycopg.connect(URL) as conn:
    conn.execute("CREATE TEMP TABLE t (id int PRIMARY KEY, title text)")
    # a) one parameter passed without the tuple comma
    try:
        conn.execute("INSERT INTO t VALUES (%s, %s)", (1, "Home"))
        conn.execute("SELECT title FROM t WHERE id = %s", 1)
    except TypeError as e:
        print("a)", type(e).__name__)
    # b) an error inside conn.transaction() rolls back only that block
    try:
        with conn.transaction():
            conn.execute("INSERT INTO t VALUES (2, 'About')")
            conn.execute("INSERT INTO t VALUES (1, 'Dup')")  # id 1 exists
    except psycopg.errors.UniqueViolation:
        print("b) duplicate key")
    rows = conn.execute("SELECT id FROM t ORDER BY id").fetchall()
    print("c)", rows)

## Progress scores

Scores are 1 to 10 against the bar for a hireable FDE, where 7 means you could do it in an interview without help. They come from what the repo and your session answers show, not from what was taught. Three weeks of topics have been taught and the repo shows the first three days, so the scores have not moved since Day 7. That is a reading of the evidence, not a judgement of your ability.

| Area | Day 14 to Day 21 | Evidence (repo and sessions) | Next step |
|---|---|---|---|
| Python | 3 to 3 | Days 1 to 3 pushed; Day 4 starters untouched; Days 5 to 20 not in the repo | Day 4 today; Day 5 on Day 23 |
| TypeScript / Node | 6 to 6 | 7 years of React; not tested yet | Node services in Phase 3 |
| DSA | 2 to 2 | 4 easy problems solved; about 30 open from Days 4 to 20 | Day 4's two today; 30 minutes daily |
| SQL and databases | 1 to 1 | Six days of SQL taught; no .sql file in the repo yet | Day 15's files today, Day 18 in Week 4 |
| Backend & APIs | 1 to 1 | HTTP with requests and httpx taught on Days 6 and 13; nothing pushed | FastAPI all of Week 4 |
| LLM application engineering | 1 to 1 | Not started | Claude API in Week 5 |
| RAG | 1 to 1 | Embedding theory from the IIIT-B programme; no hands-on yet | Weeks 6 and 7, Project 1 |
| Agents & MCP | 1 to 1 | Not started | Weeks 9 and 10 |
| Evals & reliability | 1 to 1 | pytest taught on Day 10; no tests in the repo | API tests on Day 25, evals in Week 8 |
| Cloud & deployment | 1 to 1 | Git, shell and packaging taught; one commit in the repo | Docker and cloud in Weeks 10 and 11 |
| Enterprise integration | 5 to 5 | 7 years of CMS integrations; not yet shown in Python or SQL | CMS webhook endpoint on Day 26 |
| Data skills | 2 to 2 | SQL, JSONB and pandas taught; not practised in the repo | Day 20's report.py when Week 4 allows |
| Customer-facing skills | 5 to 5 | 14 years of delivery; provisional until a mock is run | SQL mock today, customer case mock on Day 28 |
| System design for AI | 1 to 1 | Not started | Week 13 |
| Job hunt & positioning | 2 to 2 | Public repo; README pass and journey post not confirmed | LinkedIn action below |

SQL now has its own row, because it is tested on its own in most FDE loops. If you have done some of this work on your laptop without pushing it, push it today and say so in the Day 21 review: the scores go up the same day. Work that lives only on your laptop is invisible to a hiring manager too.

## Plan adjustment

Day 14 added a 1-hour catch-up lane to every day of Week 3. The repo shows that the lane did not move: the six items in it are still open. I cannot see your laptop, so this may be a push problem rather than a work problem. Either way, the plan has to change, because adding a fourth week of new topics on top of unpractised ones makes each new day harder to follow.

Two options were considered. Pausing new topics for a full week would close the gap but leave Phase 1 without its API milestone. Going on with no change would repeat Week 3. Week 4 does a third thing: FastAPI stays, because it is the Phase 1 milestone and it reuses the SQL from this week, but each day has a core task of about 2 hours and an optional stretch, and the catch-up lane shrinks to only the files that Week 4 needs.

- Daily split for Week 4: 1.5h theory · 2h core task · 1h catch-up · 0.5h DSA. Stretch tasks only on a day when the core task and the catch-up item are both pushed.

- Push every day, finished or not, with a message such as "Day 22: WIP, api_design.md done, drills pending". Half-finished work in the repo counts; finished work on a laptop does not.

- Checkpoint on Day 28: if the repo still shows nothing after Day 4, Week 5 becomes a consolidation week. You rebuild one thin project end to end (CMS export, Postgres, FastAPI) instead of starting the Claude API, and job applications move from Week 8 to Week 10.

- Tell me your real hours. If 5 hours a day is not realistic, say how many are, and the plan is cut to fit. A smaller plan that is done beats a full plan that is read.

The Week 4 catch-up lane, one item at a time. Everything else from Weeks 2 and 3 is read-only for now; the ideas return in later weeks on project code.

| Catch-up item | When | Done when |
|---|---|---|
| Day 4: contact_store.py, main.py | Today | Both runs of main.py match the expected output in its docstring |
| Day 15: schema.sql, queries.sql | Today or Day 22 | All 10 queries return the expected results on seed.sql |
| Day 5: support_ticket.py | Day 23 | The SLA breach report prints; you can explain a dataclass, which Pydantic models build on |
| Day 18: load_export.py | Days 24 and 25 | The 19 given tests pass; the API on Day 24 reads the tables it fills |
| Day 19: upsert_loader.py | Day 26 | A second load changes 0 rows; the webhook on Day 26 reuses the upsert |

## Honest calibration: the ₹1 Cr target

The calibration from Day 1 holds. A package near ₹1 Cr in India is a senior offer at a top AI company or one of a few well-funded AI startups, and those loops expect shipped LLM systems, strong system design and solid coding rounds. Sixteen weeks from zero Python does not reach that bar. Sixteen weeks can reach a strong first FDE, AI engineer or solutions engineer role, with three portfolio projects and your CMS background as the edge, and ₹1 Cr stays a 12 to 24 month goal from there.

The harder fact this week is pace. The repo shows three days of work in 24 calendar days. Job applications are planned for Week 8 (around 13 Dec) and need Project 1, a working RAG app. That date is still possible, but only if Week 4 ends with a pushed FastAPI service. If it does not, the Day 28 checkpoint moves applications to Week 10, which is a better outcome than applying with an empty repo. Hiring managers for FDE roles read GitHub before they read a CV.

## Week 4 plan: HTTP, REST and FastAPI

Every FDE engagement has an API in the middle: the customer's systems call yours, yours call theirs, and a CMS sends webhooks when content changes. Week 4 teaches HTTP properly, then builds a small content API with FastAPI on top of the Postgres schema from Week 3, with tests, an API key and a webhook endpoint. By Day 27 you have the Phase 1 milestone: a working REST API with a database. Before Day 23, check that uv --version works; FastAPI is installed with uv add "fastapi[standard]".

Daily split: 1.5h theory · 2h core task · 1h catch-up · 0.5h DSA.

| Day | Date | Topic | Deliverable |
|---|---|---|---|
| 22 | 15 Nov | HTTP and REST in depth: requests and responses, methods, status codes, headers, idempotency, pagination, curl | day22/api_design.md: a REST design for the Acme content API, plus curl drills |
| 23 | 16 Nov | FastAPI first steps: routes, path and query parameters, Pydantic models, validation errors, /docs | day23: an in-memory content API with create, read, update, delete |
| 24 | 17 Nov | FastAPI with Postgres: dependencies, a connection pool, settings from environment variables, a repository layer | day24: the content API backed by the Week 3 schema |
| 25 | 18 Nov | Testing APIs and errors: TestClient, a test database, dependency overrides, consistent error responses | day25: 20 or more API tests, green |
| 26 | 19 Nov | Webhooks and API keys: receiving a CMS webhook, HMAC signatures, idempotency, an API key header, background tasks | day26: a /webhooks/cms endpoint that checks the signature and upserts into Postgres |
| 27 | 20 Nov | Phase 1 milestone: one project with uv, logging, a README, OpenAPI docs and a demo script | content-sync-api in day27/, plus a 3-minute recorded demo |
| 28 | 21 Nov | Weekly review, Phase 1 review and plan for Week 5 (Claude API) | Review report, customer case mock, checkpoint decision |

This table is also added to the plan at the top of the doc.

## Mock interview: coding (SQL)

Type Mock interview: coding (SQL) in the Forward Deployment Engineer project chat. I will play an engineer at an AI company running a 40-minute SQL screen: a small schema of entries, authors, tags and daily views, and four questions that get harder (a filter, a group-by with HAVING, a LEFT JOIN with missing rows, and a top-N per group), then a hire or no-hire verdict with specific feedback. This is what gets scored, in order:

- Clarify before you write: which columns can be NULL, whether ties matter, and what the output should look like for one example row.

- Build the query in steps out loud: FROM and JOIN first, then WHERE, then GROUP BY, then SELECT.

- Check the edge cases yourself: an entry with no tags, a NULL view count, two entries tied on views.

- Say how it would behave on 10 million rows and which index would help.

Do the mixed review above first. It covers the same traps the mock will test.

## LinkedIn and portfolio action

This week's action turns one SQL lesson into proof of work. A post about something you built, with the code linked, does more for an FDE application than a post about a course.

- Push Day 15's schema.sql and queries.sql first, so the post has something to link to.

- Write a short post (150 to 200 words) on one trap from this week, from the CMS angle you know: for example, why NOT IN with a NULL hid every untranslated entry, or how row_number() finds the latest version of every entry. Show the two-line query, say what went wrong and how you fixed it, and link the file.

- If the Day 7 journey post is not live yet, publish that one first and this one two days later.

- On your GitHub profile, pin fde-journey and add a one-line repo description.

## DSA practice (30 min)

No new problems this week. Finish the two in day04/dsa.py, which is already in your repo with signatures, examples and hints. If both are done, move on to Day 15's dsa.py (paginate, duplicate slugs).

- Top k words. top_words(text, k). Hint: count with a dict as on Day 3, then sort the (word, count) pairs with one key that puts higher counts first and breaks ties alphabetically. Think about what a tuple key with a minus sign does.

- Balanced brackets. is_balanced(s). Hint: a list works as a stack. Map each closing bracket to its opening one, and ask what must be true about the stack when the loop ends.

## Quiz (3 quick questions)

- A content lead asks for the number of published entries per locale each day, including locales with zero. Which Week 3 tools do you combine, and in what order?

- Your nightly loader crashes at row 4,000 of 5,000. What two things from Week 3 mean the database is not left half-loaded and that running it again is safe?

- A product owner says "looking up an entry by slug is slow". What do you do before you add an index, and how do you prove the index helped?

## What you achieved this week

Week 3 covered SQL from zero on PostgreSQL: tables and keys, filters and sorting, aggregates and NULL, joins and normalisation, Python with psycopg and transactions, constraints, upserts and indexes, and analytics with CTEs, window functions and JSONB. Today you measured how much of that is in your hands yet, cut the catch-up lane down to what Week 4 needs, and set up the week that closes Phase 1.

Tomorrow (Day 22): HTTP and REST in depth: what a request and a response contain, methods, status codes, headers and idempotency, and a REST design for the Acme content API.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 21 review": your checklist with yes or no on each line, your recap answers, your seven predictions next to the real outputs, the link to your latest commit (Day 4 finished, Day 15 files in), your quiz answers, and how many hours a day you can really study next week. Then run the SQL mock in the same chat.

## Recap answers

*Your answers:*
